In [2]:
class A:
    pass


class B(A):
    pass


class C(A):
    pass


class D(B, C):
    pass


print(D.__mro__)

(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


In [5]:
D.__str__

<slot wrapper '__str__' of 'object' objects>

In [6]:
D.__str__()

TypeError: descriptor '__str__' of 'object' object needs an argument

In [ ]:
d = D()

In [8]:
d.__str__

<method-wrapper '__str__' of D object at 0x00000288CCE06660>

In [9]:
d.__str__()

'<__main__.D object at 0x00000288CCE06660>'

In [10]:
print(d)

In [11]:
print(D)

<class '__main__.D'>


In [12]:
str(D)

"<class '__main__.D'>"

In [14]:
D.__str__

<slot wrapper '__str__' of 'object' objects>

In [15]:
id(D.__str__)

2786471526704

In [19]:
hex_val = "0x00000288CCE06660"

print(int(hex_val, 16))

2786576066144


In [21]:
id(d.__str__)

2786575825216

In [23]:
D.__str__(d)

'<__main__.D object at 0x00000288CCE06660>'

In [24]:
d.__str__()

'<__main__.D object at 0x00000288CCE06660>'

Yes, it works. But it is not calling D.__str__().

Here is the exact C-engine difference:

D.__str__() (Your previous crash): You explicitly forced Python to grab the method from D's blueprint. It grabbed the raw method meant for instances of D, and it crashed because you didn't provide a self instance.

str(D) (Your new image): The built-in str() function has a hardcoded C-level rule: It completely ignores the object's own dictionary. Instead, it looks at the class that created the object.

Since D is a class, it was created by the metaclass type.

When you run str(D), the C-engine secretly executes type.__str__(D). Because D is a valid instance of type, the metaclass successfully returns the string <class '__main__.D'>.

You are not looking at the raw C-functions. You are looking at the disposable boxes Python built to hold them.

Here is exactly what those three different IDs in your image represent:

1. id(D.__str__) -> 278647...
This is the memory address of the <slot wrapper>. It is a permanent Python object stored in the class dictionary whose only job is to point to the raw C-code in the ultimate object base class.

2. The Hex Value -> 27865760...
This is the memory address of your actual instance d. This is where your data lives.

3. id(d.__str__) -> 27865758...
This is the memory address of the <method-wrapper>. This is the temporary "Bound Method" bridge.

Why are 1 and 3 different?
Because id() does not give you the memory address of the underlying C-function. It gives you the address of the wrapper object.

When you type d.__str__, Python literally asks the OS for RAM to build a brand-new <method-wrapper> object at that exact millisecond. Its only purpose is to lock your instance (ID 2) to the C-function (ID 1). Because it is a freshly built object, it gets a brand-new memory address.

(Fun/infuriating fact: If you run id(d.__str__) == id(d.__str__), Python literally builds two separate wrappers, and their IDs might not even match!)

How do you actually compare them?
For core C-level functions like __str__, you can't. CPython intentionally hides the raw C-memory pointers for built-in types for security and stability reasons. You cannot extract the pure C-address to compare them in standard Python.

However, if you write your own standard Python method, Python gives you a backdoor to strip away the temporary wrapper and look at the raw function underneath using the .__func__ attribute:

In [ ]:
class MyClass:
    def custom_method(self):
        pass


obj = MyClass()

# Strip the wrapper from the instance to see the raw function!
print(id(obj.custom_method.__func__) == id(MyClass.custom_method))
# Output: True

True


1. What you are looking for (The "Raw Code" Address)
You are looking for the memory address where the C compiler actually stored the binary instructions that run the __str__ logic.

The Reality: Python protects this. Even if you get an id(), that is not the memory address of the binary code. The interpreter does not give you access to the pointer of the actual machine instructions because that would be a security and stability nightmare.

2. What id(D.__str__) actually gives you (The "Wrapper" Object)
When you ask for D.__str__, Python cannot give you the raw machine code. Instead, it creates a Python Object on the fly.

This object is a slot_wrapper.

It is a container, like a remote control.

When you run D.__str__(), this remote control knows how to reach out to the hidden C-binary code and tell it to execute.

The "Inconsistency" you feel is actually a design choice:

Python gives you id() of the remote control (the wrapper object).

Python hides the address of the actual machine instructions (the C binary).

So, when I said "It gives you the raw function," I meant "It gives you the Python-accessible handle to that function." You are looking at the handle, not the function itself.

Why does this matter for your audit?
If you need to explain this technically:

"In CPython, built-in methods are implemented as machine-code binary functions. Because these reside outside the Python object system, the interpreter does not expose their raw memory pointers. Instead, the interpreter wraps these functions in 'wrapper' objects (like slot_wrapper or method_wrapper). When we call id(), we are getting the memory address of the wrapper object, not the address of the underlying machine instructions."


Proof: If you run id(d.__str__) ten times, you will get ten different IDs. It is building a brand-new "disposable plastic box" every time you ask for it.

In [25]:
id(d.__str__)

2786577397504

In [26]:
id(d.__str__)

2786575485536

In [27]:
id(D.__str__)

2786471526704

In [28]:
id(D.__str__)

2786471526704

In [ ]:
class E(B, C):
    pass

In [30]:
print(E)

<class '__main__.E'>


In [31]:
E.__str__

<slot wrapper '__str__' of 'object' objects>

In [32]:
id(E.__str__)

2786471526704

In [35]:
id(D.__str__)

2786471526704

In [36]:
object.__str__

<slot wrapper '__str__' of 'object' objects>

In [37]:
id(object.__str__)

2786471526704